# The HTTP API, hooks and the inbox

`amoorion api` serves every method at `POST /methods/<name>` and incoming
webhooks at `POST /hooks/<name>`. Here the app runs in-process through
FastAPI's test client, so nothing listens on a port.

In [1]:
import os

from amoorion import Amoorion

# Examples stay off your real setup: see Amoorion.init
Amoorion.init()

# The API's own token and the generic hook's token, as the vault would hold them.
os.environ["AMOORION_SECRET_API_TOKEN"] = "api-example-token"

os.environ["AMOORION_SECRET_HOOKS_GENERIC_TOKEN"] = "hook-example-token"

In [2]:
from fastapi.testclient import TestClient

from amoorion.api import create_app

client = TestClient(create_app())

client.get("/").json()

{'name': 'amoorion',
 'version': '0.1.0',
 'methods': ['accounts', 'inbox', 'plugins', 'publish', 'read', 'reply'],
 'hooks': ['generic']}

Method calls need the API token as a bearer token:

In [3]:
print(
    client.post(
        "/methods/publish",
        json={
            "text": "x",
            "to": ["dryrun"],
        },
    ).status_code
)

auth = {"Authorization": "Bearer api-example-token"}

client.post(
    "/methods/publish",
    json={
        "text": "posted over HTTP",
        "to": ["dryrun"],
    },
    headers=auth,
).json()

401


[{'platform': 'dryrun',
  'ok': True,
  'message': {'platform': 'dryrun',
   'text': 'posted over HTTP',
   'id': '1',
   'url': None,
   'author': None,
   'reply_to': None,
   'created_at': '2026-10-26T22:41:16.562007Z'},
  'error': None}]

A service tells amoorion that something happened through a hook. The
`generic` hook takes `{"text", "author"}` with its token in
`X-Amoorion-Token`; what it receives goes to the inbox:

In [4]:
client.post(
    "/hooks/generic",
    json={
        "text": "deploy finished",
        "author": "ci",
    },
    headers={"X-Amoorion-Token": "hook-example-token"},
).json()

{'platform': 'hook:generic',
 'text': 'deploy finished',
 'id': 'd69a21ae84f34250859c2514de5a2720',
 'url': None,
 'author': {'platform': 'hook:generic', 'handle': 'ci', 'name': None},
 'reply_to': None,
 'created_at': None}

In [5]:
client.post(
    "/methods/inbox",
    json={},
    headers=auth,
).json()

[{'platform': 'hook:generic',
  'text': 'deploy finished',
  'id': 'd69a21ae84f34250859c2514de5a2720',
  'url': None,
  'author': {'platform': 'hook:generic', 'handle': 'ci', 'name': None},
  'reply_to': None,
  'created_at': None}]